# Prepare the candidate dataset

This notebook will read `candidates.parquet` and create the prepared train, validation, and test Parquet files.

## Import dependencies

This block imports only the libraries needed by the preparation notebook. It does not read files or change data.

In [ ]:
from datetime import UTC, datetime
from pathlib import Path

import pyarrow.parquet as pq

## Configure the input, output, and date bounds

Set the candidate Parquet location, the directory for the prepared splits, and optional timezone-aware UTC datetime bounds. The `since` bound is inclusive; the `until` bound is exclusive.

In [ ]:
candidates_path = Path("../candidates.parquet")
output_directory = Path("prepared")
since = None  # Example: datetime(2024, 1, 1, tzinfo=UTC)
until = None  # Example: datetime(2025, 1, 1, tzinfo=UTC)

## Load the candidate Parquet file

In [ ]:
if not candidates_path.is_file():
    raise FileNotFoundError(candidates_path)

candidate_table = pq.read_table(candidates_path)

## Validate the candidate schema

The preparation step expects the six canonical candidate columns. This block checks that they exist, selects them in a stable order, and converts the table into Python rows for the later preparation steps.

In [ ]:
candidate_columns = (
    "id",
    "repository_group_id",
    "subject",
    "diff",
    "committed_at_utc",
    "patch_id",
)
missing_columns = [
    column for column in candidate_columns if column not in candidate_table.column_names
]
if missing_columns:
    raise ValueError(f"Candidate Parquet is missing columns: {', '.join(missing_columns)}")

selected_table = candidate_table.select(list(candidate_columns))
rows = selected_table.to_pylist()
source_count = len(rows)

## Filter rows by the committed date

This block keeps rows where `since <= committed_at_utc < until`. Leaving either bound as `None` makes that side of the date window open.

In [ ]:
def committed_at_utc(row: dict[str, object]) -> datetime:
    return datetime.fromisoformat(str(row["committed_at_utc"]).replace("Z", "+00:00")).astimezone(
        UTC
    )


filtered_rows = [
    row
    for row in rows
    if (since is None or since <= committed_at_utc(row))
    and (until is None or committed_at_utc(row) < until)
]
filtered_count = len(filtered_rows)

## Sort and deduplicate patch groups

Rows are ordered by commit time, with `id` as a deterministic tie-breaker. When multiple rows share a `patch_id`, only the earliest row is kept so that one patch group cannot appear in multiple splits.

In [ ]:
filtered_rows.sort(key=lambda row: (committed_at_utc(row), str(row["id"])))

seen_patch_ids = set()
deduplicated_rows = []
for row in filtered_rows:
    patch_id = str(row["patch_id"])
    group_id = patch_id or f"id:{row['id']}"
    if group_id in seen_patch_ids:
        continue
    seen_patch_ids.add(group_id)
    deduplicated_rows.append(row)

deduplicated_count = len(deduplicated_rows)